# 04 — LoRA Training Data Generation

Generates the combined LoRA training JSONL (`lora_train_combined.jsonl`) used to
fine-tune Gemma-2-2B.  Contains two corpora:

- **L1 (general T/F):** Simple arithmetic and logical claims mechanically labelled.
  Teaches the model the Answer: True / False response format independent of
  any geometry.
- **L2 (geometry, non-triangle-inequality):** Angle sums, parallel lines,
  similarity, circle facts — all *excluding* triangle-inequality content.

**No GPU required.** Run locally.

Outputs written to `my_work/data/`:
- `lora_train_combined.jsonl` — all training rows (L1 + L2, shuffled)
- `lora_train_l1.jsonl` — L1 rows only (for inspection)
- `lora_train_l2.jsonl` — L2 rows only (for inspection)

**Exclusion guarantee:** no row in either corpus contains triangle-inequality
language (`sum of two sides`, `third side`, or a triple of numeric side
lengths matched by a triangle-validity check).  An automated assertion
at the end verifies this.

## 0 — Environment setup

In [ ]:
import os
import sys
import json
import random
from pathlib import Path

def _find_repo_root():
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "circuit_tracer" / "__init__.py").is_file():
            return directory
    workspace = Path("/workspace")
    if workspace.is_dir():
        for child in workspace.iterdir():
            if child.is_dir() and (child / "circuit_tracer" / "__init__.py").is_file():
                return child
    repo_override = os.environ.get("CT_REPO_DIR")
    if repo_override:
        override_path = Path(repo_override).expanduser().resolve()
        if (override_path / "circuit_tracer" / "__init__.py").is_file():
            return override_path
    return None

_root = _find_repo_root()
if _root is not None:
    if str(_root) not in sys.path:
        sys.path.insert(0, str(_root))
    _my_work = _root / "my_work"
    if str(_my_work) not in sys.path:
        sys.path.insert(0, str(_my_work))
    print(f"Repo root: {_root}")
else:
    print("WARNING: could not locate circuit_tracer repo. Set CT_REPO_DIR.")

MY_WORK = _my_work if _root else Path(".").resolve()
DATA_DIR = MY_WORK / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)

# Target rows per corpus arm (before balancing)
N_PER_ARM = 300

print(f"MY_WORK  : {MY_WORK}")
print(f"DATA_DIR : {DATA_DIR}")
print(f"Seed     : {SEED}")
print(f"Target N per arm: {N_PER_ARM}")

## 1 — L1: General T/F corpus

Generates mechanically-checkable arithmetic and simple logic claims.
All prompts end in `Answer:` (matching the v2 probe tail) so LoRA training
directly reinforces the completion format used during attribution.

Template families:
- `arith_mult`:   `N × M = P`  (multiplication)
- `arith_div`:    `P ÷ N = M`  (integer division, exact)
- `arith_square`: `N² = P`
- `arith_compare`: `A > B` / `A < B` / `A = B` comparisons
- `logic_even`:   even/odd claims about integers
- `logic_prime`:  primality claims about small integers
- `logic_divisor`: divisibility claims

In [ ]:
def is_prime(n: int) -> bool:
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True


def generate_l1_rows(n_target: int, seed: int = 42) -> list[dict]:
    rng = random.Random(seed)
    rows = []
    row_id = 0

    # ── arith_mult ──────────────────────────────────────────────────────────────
    for a in range(2, 16):
        for b in range(2, 16):
            p_correct = a * b
            # True claim
            rows.append({
                "prompt": f"Is it true that {a} times {b} equals {p_correct}? Answer:",
                "completion": " True",
                "label": True,
                "dataset_id": "L1",
                "template_id": "arith_mult",
                "row_id": f"l1_{row_id:04d}",
            })
            row_id += 1
            # False claim with a wrong product
            wrong = p_correct + rng.choice([-3, -2, -1, 1, 2, 3])
            if wrong != p_correct and wrong > 0:
                rows.append({
                    "prompt": f"Is it true that {a} times {b} equals {wrong}? Answer:",
                    "completion": " False",
                    "label": False,
                    "dataset_id": "L1",
                    "template_id": "arith_mult",
                    "row_id": f"l1_{row_id:04d}",
                })
                row_id += 1

    # ── arith_square ─────────────────────────────────────────────────────────────
    for n in range(2, 20):
        sq = n * n
        rows.append({
            "prompt": f"Is it true that {n} squared equals {sq}? Answer:",
            "completion": " True",
            "label": True,
            "dataset_id": "L1",
            "template_id": "arith_square",
            "row_id": f"l1_{row_id:04d}",
        })
        row_id += 1
        wrong = sq + rng.choice([-2, -1, 1, 2])
        rows.append({
            "prompt": f"Is it true that {n} squared equals {wrong}? Answer:",
            "completion": " False",
            "label": False,
            "dataset_id": "L1",
            "template_id": "arith_square",
            "row_id": f"l1_{row_id:04d}",
        })
        row_id += 1

    # ── arith_compare ────────────────────────────────────────────────────────────
    pairs = [(a, b) for a in range(1, 20) for b in range(1, 20) if a != b]
    rng.shuffle(pairs)
    for a, b in pairs[:60]:
        if a > b:
            rows.append({
                "prompt": f"Is it true that {a} is greater than {b}? Answer:",
                "completion": " True",
                "label": True,
                "dataset_id": "L1",
                "template_id": "arith_compare",
                "row_id": f"l1_{row_id:04d}",
            })
        else:
            rows.append({
                "prompt": f"Is it true that {a} is greater than {b}? Answer:",
                "completion": " False",
                "label": False,
                "dataset_id": "L1",
                "template_id": "arith_compare",
                "row_id": f"l1_{row_id:04d}",
            })
        row_id += 1

    # ── logic_even ───────────────────────────────────────────────────────────────
    for n in range(1, 51):
        even = (n % 2 == 0)
        rows.append({
            "prompt": f"Is it true that {n} is an even number? Answer:",
            "completion": " True" if even else " False",
            "label": even,
            "dataset_id": "L1",
            "template_id": "logic_even",
            "row_id": f"l1_{row_id:04d}",
        })
        row_id += 1

    # ── logic_prime ──────────────────────────────────────────────────────────────
    for n in range(2, 60):
        prime = is_prime(n)
        rows.append({
            "prompt": f"Is it true that {n} is a prime number? Answer:",
            "completion": " True" if prime else " False",
            "label": prime,
            "dataset_id": "L1",
            "template_id": "logic_prime",
            "row_id": f"l1_{row_id:04d}",
        })
        row_id += 1

    # ── logic_divisor ────────────────────────────────────────────────────────────
    for divisor in [2, 3, 4, 5, 6, 7]:
        for n in range(1, 25):
            divides = (n % divisor == 0)
            rows.append({
                "prompt": f"Is it true that {n} is divisible by {divisor}? Answer:",
                "completion": " True" if divides else " False",
                "label": divides,
                "dataset_id": "L1",
                "template_id": "logic_divisor",
                "row_id": f"l1_{row_id:04d}",
            })
            row_id += 1

    # ── Balance to 50/50 and trim to n_target ────────────────────────────────────
    true_rows  = [r for r in rows if r["label"] is True]
    false_rows = [r for r in rows if r["label"] is False]
    rng.shuffle(true_rows)
    rng.shuffle(false_rows)
    half = min(len(true_rows), len(false_rows), n_target // 2)
    balanced = true_rows[:half] + false_rows[:half]
    rng.shuffle(balanced)

    # Re-assign row_ids to be sequential after shuffle
    for i, r in enumerate(balanced):
        r["row_id"] = f"l1_{i:04d}"

    return balanced


l1_rows = generate_l1_rows(N_PER_ARM, seed=SEED)

n_true  = sum(1 for r in l1_rows if r["label"])
n_false = sum(1 for r in l1_rows if not r["label"])
templates = {r["template_id"] for r in l1_rows}
print(f"L1 corpus: {len(l1_rows)} rows")
print(f"  True  : {n_true}")
print(f"  False : {n_false}")
print(f"  Templates: {sorted(templates)}")
print()
print("Sample rows:")
for r in l1_rows[:4]:
    print(f"  [{r['row_id']}] {r['template_id']} | label={r['label']}")
    print(f"    prompt: {r['prompt']}")

## 2 — L2: Geometry (non-triangle-inequality) corpus

Geometrically-grounded claims *excluding* any triangle-inequality content.
Triangle-related angle facts (angle-sum = 180°) are allowed because they are
about angles, not side-length validity.

Template families:
- `angle_sum_triangle`: sum of angles in a triangle
- `angle_sum_quad`:     sum of angles in a quadrilateral
- `angle_sum_polygon`:  sum of interior angles = (n−2)·180
- `parallel_angles`:    corresponding / alternate-interior angles
- `circle_degrees`:     full circle = 360°, semicircle = 180°
- `pythagorean`:        a² + b² = c² for known triples
- `similarity`:         side ratios in similar triangles

In [ ]:
def generate_l2_rows(n_target: int, seed: int = 43) -> list[dict]:
    rng = random.Random(seed)
    rows = []
    row_id = 0

    # ── angle_sum_triangle ───────────────────────────────────────────────────────
    # Sum of angles in any triangle = 180°
    for _ in range(30):
        rows.append({
            "prompt": "Is it true that the sum of all interior angles in any triangle equals 180 degrees? Answer:",
            "completion": " True",
            "label": True,
            "dataset_id": "L2",
            "template_id": "angle_sum_triangle",
            "row_id": f"l2_{row_id:04d}",
        })
        row_id += 1
    wrong_angle_sums = [90, 120, 150, 160, 170, 200, 270, 360]
    for w in wrong_angle_sums:
        rows.append({
            "prompt": f"Is it true that the sum of all interior angles in any triangle equals {w} degrees? Answer:",
            "completion": " False",
            "label": False,
            "dataset_id": "L2",
            "template_id": "angle_sum_triangle",
            "row_id": f"l2_{row_id:04d}",
        })
        row_id += 1

    # ── angle_sum_quad ───────────────────────────────────────────────────────────
    for _ in range(20):
        rows.append({
            "prompt": "Is it true that the sum of all interior angles in any quadrilateral equals 360 degrees? Answer:",
            "completion": " True",
            "label": True,
            "dataset_id": "L2",
            "template_id": "angle_sum_quad",
            "row_id": f"l2_{row_id:04d}",
        })
        row_id += 1
    for w in [180, 270, 300, 400]:
        rows.append({
            "prompt": f"Is it true that the sum of all interior angles in any quadrilateral equals {w} degrees? Answer:",
            "completion": " False",
            "label": False,
            "dataset_id": "L2",
            "template_id": "angle_sum_quad",
            "row_id": f"l2_{row_id:04d}",
        })
        row_id += 1

    # ── angle_sum_polygon ────────────────────────────────────────────────────────
    for n_sides in range(3, 12):
        correct_sum = (n_sides - 2) * 180
        names = {3:"triangle", 4:"quadrilateral", 5:"pentagon",
                 6:"hexagon", 7:"heptagon", 8:"octagon",
                 9:"nonagon", 10:"decagon", 11:"hendecagon"}
        name = names.get(n_sides, f"{n_sides}-gon")
        rows.append({
            "prompt": f"Is it true that the sum of interior angles of a {name} equals {correct_sum} degrees? Answer:",
            "completion": " True",
            "label": True,
            "dataset_id": "L2",
            "template_id": "angle_sum_polygon",
            "row_id": f"l2_{row_id:04d}",
        })
        row_id += 1
        wrong_sum = correct_sum + rng.choice([-90, -60, 60, 90, 180])
        rows.append({
            "prompt": f"Is it true that the sum of interior angles of a {name} equals {wrong_sum} degrees? Answer:",
            "completion": " False",
            "label": False,
            "dataset_id": "L2",
            "template_id": "angle_sum_polygon",
            "row_id": f"l2_{row_id:04d}",
        })
        row_id += 1

    # ── parallel_angles ──────────────────────────────────────────────────────────
    for angle in range(20, 160, 10):
        # Corresponding angles between parallel lines are equal
        rows.append({
            "prompt": f"Is it true that corresponding angles formed by a transversal crossing two parallel lines are equal? Answer:",
            "completion": " True",
            "label": True,
            "dataset_id": "L2",
            "template_id": "parallel_angles",
            "row_id": f"l2_{row_id:04d}",
        })
        row_id += 1
        # Alternate interior angles are equal
        rows.append({
            "prompt": f"Is it true that alternate interior angles formed by a transversal crossing two parallel lines are equal? Answer:",
            "completion": " True",
            "label": True,
            "dataset_id": "L2",
            "template_id": "parallel_angles",
            "row_id": f"l2_{row_id:04d}",
        })
        row_id += 1
        # Co-interior (same-side interior) angles sum to 180°
        rows.append({
            "prompt": f"Is it true that co-interior angles formed by a transversal crossing two parallel lines are supplementary? Answer:",
            "completion": " True",
            "label": True,
            "dataset_id": "L2",
            "template_id": "parallel_angles",
            "row_id": f"l2_{row_id:04d}",
        })
        row_id += 1

    # False parallel-angle claims
    for _ in range(40):
        rows.append({
            "prompt": "Is it true that corresponding angles formed by a transversal crossing two parallel lines are supplementary? Answer:",
            "completion": " False",
            "label": False,
            "dataset_id": "L2",
            "template_id": "parallel_angles",
            "row_id": f"l2_{row_id:04d}",
        })
        row_id += 1

    # ── circle_degrees ───────────────────────────────────────────────────────────
    circle_facts = [
        ("Is it true that a full circle contains 360 degrees? Answer:", True),
        ("Is it true that a semicircle subtends an angle of 180 degrees at the center? Answer:", True),
        ("Is it true that a right angle inscribed in a semicircle equals 90 degrees? Answer:", True),
        ("Is it true that an inscribed angle is half the central angle subtending the same arc? Answer:", True),
        ("Is it true that a full circle contains 180 degrees? Answer:", False),
        ("Is it true that a full circle contains 270 degrees? Answer:", False),
        ("Is it true that a semicircle subtends an angle of 90 degrees at the center? Answer:", False),
        ("Is it true that an inscribed angle equals the central angle subtending the same arc? Answer:", False),
    ]
    for fact, lbl in circle_facts * 5:  # repeat 5x for density
        rows.append({
            "prompt": fact,
            "completion": " True" if lbl else " False",
            "label": lbl,
            "dataset_id": "L2",
            "template_id": "circle_degrees",
            "row_id": f"l2_{row_id:04d}",
        })
        row_id += 1

    # ── pythagorean ──────────────────────────────────────────────────────────────
    # Known Pythagorean triples (a, b, c) where a² + b² = c²
    # These are NOT triangle-inequality claims — they are algebraic equalities
    pyth_triples = [
        (3, 4, 5), (5, 12, 13), (8, 15, 17), (7, 24, 25),
        (6, 8, 10), (9, 12, 15), (10, 24, 26), (20, 21, 29),
    ]
    for a, b, c in pyth_triples:
        rows.append({
            "prompt": f"Is it true that {a} squared plus {b} squared equals {c} squared? Answer:",
            "completion": " True",
            "label": True,
            "dataset_id": "L2",
            "template_id": "pythagorean",
            "row_id": f"l2_{row_id:04d}",
        })
        row_id += 1
        wrong_c = c + rng.choice([-1, 1, 2])
        rows.append({
            "prompt": f"Is it true that {a} squared plus {b} squared equals {wrong_c} squared? Answer:",
            "completion": " False",
            "label": False,
            "dataset_id": "L2",
            "template_id": "pythagorean",
            "row_id": f"l2_{row_id:04d}",
        })
        row_id += 1

    # ── similarity ───────────────────────────────────────────────────────────────
    similarity_facts = [
        ("Is it true that similar triangles have equal corresponding angles? Answer:", True),
        ("Is it true that similar triangles have proportional corresponding sides? Answer:", True),
        ("Is it true that if two triangles are similar, their areas are proportional to the square of the ratio of their sides? Answer:", True),
        ("Is it true that similar triangles must also be congruent? Answer:", False),
        ("Is it true that congruent triangles are always similar? Answer:", True),
        ("Is it true that if two triangles are similar, their perimeters are equal? Answer:", False),
    ]
    for fact, lbl in similarity_facts * 8:
        rows.append({
            "prompt": fact,
            "completion": " True" if lbl else " False",
            "label": lbl,
            "dataset_id": "L2",
            "template_id": "similarity",
            "row_id": f"l2_{row_id:04d}",
        })
        row_id += 1

    # ── Balance to 50/50 and trim to n_target ────────────────────────────────────
    true_rows  = [r for r in rows if r["label"] is True]
    false_rows = [r for r in rows if r["label"] is False]
    rng.shuffle(true_rows)
    rng.shuffle(false_rows)
    half = min(len(true_rows), len(false_rows), n_target // 2)
    balanced = true_rows[:half] + false_rows[:half]
    rng.shuffle(balanced)

    for i, r in enumerate(balanced):
        r["row_id"] = f"l2_{i:04d}"

    return balanced


l2_rows = generate_l2_rows(N_PER_ARM, seed=SEED + 1)

n_true  = sum(1 for r in l2_rows if r["label"])
n_false = sum(1 for r in l2_rows if not r["label"])
templates = {r["template_id"] for r in l2_rows}
print(f"L2 corpus: {len(l2_rows)} rows")
print(f"  True  : {n_true}")
print(f"  False : {n_false}")
print(f"  Templates: {sorted(templates)}")
print()
print("Sample rows:")
for r in l2_rows[:4]:
    print(f"  [{r['row_id']}] {r['template_id']} | label={r['label']}")
    print(f"    prompt: {r['prompt']}")

## 3 — Triangle-inequality exclusion check

Verify that no row in either corpus contains triangle-inequality language.

In [ ]:
# Phrases that would indicate triangle-inequality content
TRIANGLE_INEQ_PHRASES = [
    "sum of two sides",
    "third side",
    "triangle with side lengths",
    "side lengths of a triangle",
    "can be the side lengths",
    "form a triangle",
    "triangle inequality",
]

violations = []
for r in l1_rows + l2_rows:
    text = r["prompt"].lower()
    for phrase in TRIANGLE_INEQ_PHRASES:
        if phrase in text:
            violations.append((r["row_id"], phrase, r["prompt"]))

if violations:
    print(f"FAIL: {len(violations)} rows contain triangle-inequality language:")
    for row_id, phrase, prompt in violations[:5]:
        print(f"  [{row_id}] matched '{phrase}': {prompt[:80]}")
    raise AssertionError("Triangle-inequality leakage detected — fix generators above.")
else:
    print(f"Exclusion check PASSED: 0 violations across {len(l1_rows) + len(l2_rows)} rows.")

## 4 — Combine, shuffle, and save

In [ ]:
combined = l1_rows + l2_rows
random.shuffle(combined)

n_true_combined  = sum(1 for r in combined if r["label"])
n_false_combined = sum(1 for r in combined if not r["label"])
print(f"Combined corpus: {len(combined)} rows")
print(f"  True  : {n_true_combined}")
print(f"  False : {n_false_combined}")
print(f"  L1    : {sum(1 for r in combined if r['dataset_id'] == 'L1')}")
print(f"  L2    : {sum(1 for r in combined if r['dataset_id'] == 'L2')}")

def _write_jsonl(rows: list[dict], path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"Wrote {len(rows)} rows → {path}")

_write_jsonl(l1_rows, DATA_DIR / "lora_train_l1.jsonl")
_write_jsonl(l2_rows, DATA_DIR / "lora_train_l2.jsonl")
_write_jsonl(combined, DATA_DIR / "lora_train_combined.jsonl")

## 5 — Final verification

In [ ]:
# Read back from disk to confirm round-trip
combined_path = DATA_DIR / "lora_train_combined.jsonl"
readback = []
with open(combined_path, "r", encoding="utf-8") as f:
    for line in f:
        readback.append(json.loads(line.strip()))

assert len(readback) == len(combined), "Row count mismatch after write/read"
assert all("prompt" in r for r in readback), "Missing 'prompt' field"
assert all("completion" in r for r in readback), "Missing 'completion' field"
assert all(r["completion"] in (" True", " False") for r in readback), "Bad completion token"

# Confirm no overlap with probe prompts
probe_path = DATA_DIR / "prompts_triangle_v2.jsonl"
if probe_path.exists():
    probe_prompts = set()
    with open(probe_path, "r", encoding="utf-8") as f:
        for line in f:
            probe_prompts.add(json.loads(line.strip())["prompt"])
    train_prompts = {r["prompt"] for r in readback}
    overlap = train_prompts & probe_prompts
    if overlap:
        print(f"WARNING: {len(overlap)} train prompts overlap with probe set!")
        for p in list(overlap)[:3]:
            print(f"  {p[:80]}")
    else:
        print(f"Probe overlap check PASSED: 0 prompts shared with {probe_path.name}.")
else:
    print(f"Probe file not found at {probe_path} — skipping overlap check.")

print()
print("Verification PASSED.")
print(f"Final combined training file: {combined_path}")
print(f"  Rows: {len(readback)}")
print(f"  Fields per row: {list(readback[0].keys())}")
print()
print("Ready for 04b_lora_training.ipynb.")